# Univariate Feature Selection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_09/VL09_Univariate_Feature_Selection.ipynb)

**Lernziel:** Verstehen und anwenden von univariaten Feature Selection Methoden

**Methode:**
- 📊 **F-Test (ANOVA):** Für numerische Features und kategoriale Zielvariable

**Szenario:** Welche numerischen Features zeigen einzeln Mittelwertunterschiede zwischen den Klassen, und wie wählen wir damit Features aus?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split

# Für bessere Plots
plt.style.use('default')
sns.set_palette("husl")

print("📚 Libraries geladen!")

## 1. Datensatz erstellen

Wir erstellen einen synthetischen Datensatz mit numerischen Signal- und Rauschmerkmalen. Die zusätzlichen kategorialen Features sind nur ein Ausblick auf einen späteren Chi-Quadrat-Test; sie werden in diesem Notebook nicht ausgewertet.

In [ ]:
# Synthetischen Datensatz erstellen
np.random.seed(42)

# Basis-Features mit make_classification
X_base, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_clusters_per_class=1,
    random_state=42
)

# Zusätzliche kategoriale Features erstellen
n_samples = X_base.shape[0]

# Kategoriale Features (für Chi-Square Test)
kategorie_A = np.random.choice(['low', 'medium', 'high'], n_samples, p=[0.3, 0.4, 0.3])
kategorie_B = np.random.choice(['type1', 'type2', 'type3'], n_samples)

# Eine kategorie die mit y korreliert (informativ)
kategorie_informativ = np.where(y == 1, 
                                np.random.choice(['A', 'B'], n_samples, p=[0.8, 0.2]),
                                np.random.choice(['A', 'B'], n_samples, p=[0.2, 0.8]))

# Noise Features hinzufügen
noise_features = np.random.randn(n_samples, 5)

# Alles zu einem DataFrame kombinieren
df = pd.DataFrame(X_base, columns=[f'numeric_{i}' for i in range(10)])
df['kategorie_A'] = kategorie_A
df['kategorie_B'] = kategorie_B
df['kategorie_informativ'] = kategorie_informativ

# Noise features hinzufügen
for i in range(5):
    df[f'noise_{i}'] = noise_features[:, i]

df['target'] = y

print(f"Dataset erstellt: {df.shape}")
print(f"Features: {df.columns.tolist()}")
print(f"\nTarget-Verteilung:")
print(df['target'].value_counts())

df.head()

## 2. F-Test (ANOVA) für numerische Features

Der **F-Test** prüft für jedes numerische Feature die Nullhypothese gleicher Klassenmittelwerte. Hohe F-Werte zeigen große Mittelwertunterschiede relativ zur Streuung innerhalb der Klassen.

Wir teilen zuerst in Training und Test. Die überwachte Feature-Auswahl darf die Test-Zielwerte nicht kennen.

**Einordnung:** Die klassische ANOVA setzt unabhängige Beobachtungen, näherungsweise normalverteilte Werte innerhalb der Klassen und gleiche Klassenvarianzen voraus. Die synthetischen Daten garantieren diese Annahmen nicht vollständig; die p-Werte dienen hier der Illustration.

Bei 15 Tests sind zufällige Treffer möglich. Wir zeigen deshalb neben den unkorrigierten p-Werten eine konservative **Bonferroni-Korrektur**: `min(p * Anzahl_Tests, 1)`. Statistische Signifikanz ist weder ein Maß für Vorhersagegenauigkeit noch ein Beweis für Kausalität. Ein nicht signifikanter F-Test schließt nichtlineare Effekte oder nützliche Feature-Kombinationen nicht aus.

In [ ]:
# Numerische Features extrahieren
numeric_features = [col for col in df.columns if col.startswith(('numeric_', 'noise_'))]
X_numeric = df[numeric_features]
y_target = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y_target, test_size=0.2, stratify=y_target, random_state=42
)

print(f"Numerische Features: {len(numeric_features)}")
print(f"Features: {numeric_features}")

# k=4 ist für diese Übung vorgegeben, nicht anhand der Testdaten gewählt.
f_selector = SelectKBest(score_func=f_classif, k=4)
f_selector.fit(X_train, y_train)

# Ergebnisse sammeln
f_scores = f_selector.scores_
f_pvalues = f_selector.pvalues_

# Ergebnisse in DataFrame
f_results = pd.DataFrame({
    'Feature': numeric_features,
    'F_Score': f_scores,
    'P_Value': f_pvalues,
    'Significant': f_pvalues < 0.05,
    'P_Bonferroni': np.minimum(f_pvalues * len(numeric_features), 1),
    'Significant_Bonferroni': f_pvalues < 0.05 / len(numeric_features)
})

# Nach F-Score sortieren
f_results = f_results.sort_values('F_Score', ascending=False)

print("\n🔍 F-Test Ergebnisse (nur Training; Significant = unkorrigiert):")
print(f_results)

In [ ]:
# F-Test Ergebnisse visualisieren
plt.figure(figsize=(12, 6))

# Subplot 1: F-Scores
plt.subplot(1, 2, 1)
colors = ['green' if sig else 'red' for sig in f_results['Significant']]
bars = plt.bar(range(len(f_results)), f_results['F_Score'], color=colors, alpha=0.7)
plt.xticks(range(len(f_results)), f_results['Feature'], rotation=45, ha='right')
plt.ylabel('F-Score')
plt.title('F-Test Scores (numerische Features)')
plt.grid(True, alpha=0.3)

# Legende
import matplotlib.patches as mpatches
green_patch = mpatches.Patch(color='green', alpha=0.7, label='p < 0.05 (unkorrigiert)')
red_patch = mpatches.Patch(color='red', alpha=0.7, label='p >= 0.05 (unkorrigiert)')
plt.legend(handles=[green_patch, red_patch])

# Subplot 2: P-Values
plt.subplot(1, 2, 2)
plt.bar(range(len(f_results)), -np.log10(f_results['P_Value'].clip(lower=np.finfo(float).tiny)), color=colors, alpha=0.7)
plt.xticks(range(len(f_results)), f_results['Feature'], rotation=45, ha='right')
plt.ylabel('-log10(P-Value)')
plt.title('Statistische Signifikanz')
plt.axhline(y=-np.log10(0.05), color='black', linestyle='--', alpha=0.7, label='p = 0.05 (unkorrigiert)')
plt.axhline(y=-np.log10(0.05 / len(numeric_features)), color='purple', linestyle=':', label='Bonferroni-Grenze')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Interpretation:")
print("- Farben beziehen sich auf unkorrigierte p-Werte, nicht auf die Top-k-Auswahl")
print("- Höhere F-Scores: stärkere einzelne Mittelwertunterschiede relativ zur Streuung")
print("- Signifikanz ist nicht gleich Vorhersagequalität; nicht signifikant bedeutet nicht nutzlos")

## 3. Vier Features auswählen und beide Datensätze transformieren

`SelectKBest(k=4)` behält genau die vier höchsten Training-F-Scores, unabhängig von einer p-Wert-Grenze. `k='all'` würde nur bewerten und keine Features entfernen.

Nur auf Training **fitten**, auf Training und Test **transformieren**. Soll `k` optimiert werden, geschieht das innerhalb der Trainingsdaten per Kreuzvalidierung, mit dem Selektor in einer ML-Pipeline. Der Testdatensatz bleibt bis zur abschließenden Modellbewertung unberührt.

In [ ]:
selected_features = f_selector.get_feature_names_out()
X_train_selected = f_selector.transform(X_train)
X_test_selected = f_selector.transform(X_test)

print('Ausgewählte Features:', selected_features.tolist())
print('Training:', X_train.shape, '->', X_train_selected.shape)
print('Test:', X_test.shape, '->', X_test_selected.shape)
assert X_train_selected.shape == (len(X_train), 4)
assert X_test_selected.shape == (len(X_test), 4)
np.testing.assert_allclose(X_train_selected, X_train[selected_features].to_numpy())
np.testing.assert_allclose(X_test_selected, X_test[selected_features].to_numpy())

## Zusammenfassung: F-Test für Feature Selection

### Was haben wir gelernt?

**F-Test (ANOVA):**
- 📊 Misst, ob numerische Features signifikant unterschiedliche Mittelwerte zwischen Klassen haben
- 🎯 Höhere F-Scores zeigen stärkere standardisierte Mittelwertunterschiede, nicht automatisch bessere Prognosen
- 📈 p < 0.05 gilt hier pro Test; bei mehreren Tests die Fehlerhäufung berücksichtigen

**Praktische Anwendung:**
- ✅ Gut für: Numerische Features → Kategoriale Zielvariable
- 🔍 Hilft bei: Auswahl einzeln auffälliger Features als Ausgangspunkt für Modellierung
- ⚠️ Limitierung: Betrachtet Features einzeln (univariat), nicht Kombinationen
- Nicht signifikante Features können trotzdem nützlich sein
- Auswahl nur auf Training fitten; dieselben Spalten auf Test anwenden

**Next Steps:**
- Chi-Square Test für kategoriale Features
- Multivariate Feature Selection Methoden
- Anwendung in echten ML-Pipelines